# 17 — Bond Forwards & Repo

- **Bond forward pricing** (dirty / clean / NPV)
- Implied repo rate
- Spot income and carry cost
- AD: sensitivity to discount rate and bond yield

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.instruments.forward import BondForward
from ql_jax.engines.bond.discounting import (
    bond_forward_dirty_price, bond_forward_clean_price,
    bond_forward_npv, bond_forward_spot_income)
from ql_jax.instruments.bond import make_fixed_rate_bond
from ql_jax.termstructures.yield_.flat_forward import FlatForward
from ql_jax.time.date import Date

# Parameters
today = Date(15, 1, 2024)
delivery = Date(15, 7, 2024)   # 6 months forward
maturity = Date(15, 1, 2034)   # 10-year bond

COUPON = 0.05
FACE = 100.0
REPO_RATE = 0.04
BOND_YIELD = 0.05

---
## 1. Create Bond and Curves

In [ ]:
bond = make_fixed_rate_bond(
    face_value=FACE, coupon_rate=COUPON,
    issue_date=today, maturity_date=maturity,
    settlement_days=0, frequency=2)

repo_curve = FlatForward(today, REPO_RATE)
bond_curve = FlatForward(today, BOND_YIELD)

print(f"Bond coupon : {COUPON*100:.1f}%")
print(f"Repo rate   : {REPO_RATE*100:.1f}%")
print(f"Bond yield  : {BOND_YIELD*100:.1f}%")

---
## 2. Forward Pricing

In [ ]:
dirty_fwd = float(bond_forward_dirty_price(
    bond, delivery, repo_curve, bond_curve=bond_curve, settlement_date=today))

clean_fwd = float(bond_forward_clean_price(
    bond, delivery, repo_curve, bond_curve=bond_curve, settlement_date=today))

print(f"Forward dirty price : {dirty_fwd:.6f}")
print(f"Forward clean price : {clean_fwd:.6f}")

---
## 3. Forward NPV

In [ ]:
# Long position: NPV at different strikes
strikes = np.linspace(90, 110, 20)
npvs = [float(bond_forward_npv(
    bond, delivery, k, repo_curve, bond_curve=bond_curve,
    settlement_date=today, position=1)) for k in strikes]

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(strikes, npvs, 'b-', linewidth=2)
plt.axhline(0, color='gray', ls='--')
plt.axvline(dirty_fwd, color='red', ls='--', alpha=0.7, label=f'Fair fwd = {dirty_fwd:.2f}')
plt.xlabel('Forward Strike')
plt.ylabel('NPV')
plt.title('Bond Forward NPV (Long Position)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 4. AD: Rate Sensitivities

In [ ]:
def fwd_price_wrt_repo(repo_r):
    rc = FlatForward(today, repo_r)
    return bond_forward_dirty_price(bond, delivery, rc, bond_curve=bond_curve, settlement_date=today)

def fwd_price_wrt_yield(bond_y):
    bc = FlatForward(today, bond_y)
    return bond_forward_dirty_price(bond, delivery, repo_curve, bond_curve=bc, settlement_date=today)

dP_drepo = float(jax.grad(fwd_price_wrt_repo)(REPO_RATE))
dP_dyield = float(jax.grad(fwd_price_wrt_yield)(BOND_YIELD))

print(f"dP/d(repo)   : {dP_drepo:.4f}")
print(f"dP/d(yield)  : {dP_dyield:.4f}")

---
## 5. Exercises

1. **Carry trade**: Plot the forward price as a function of repo rate. At what repo rate is carry zero?
2. **Coupon reinvestment**: Compare forward pricing with and without income (coupon between now and delivery).
3. **Implied repo**: Given a market forward price, back out the implied repo rate using `jax.grad`-based root finding.

---
*End of Notebook 17*